In [9]:
################ Import necessary libraries
import pandas as pd
import numpy as np
import glob

In [10]:
################ Import processed data
market_share = pd.read_excel("C:/Users/Lenovo/Desktop/Dissertaion/Code/MSc-Dissertation/final_cleaned_data.xlsx")

# Import the steel price data
steel_price_file = pd.read_excel(
    "C:/Users/Lenovo/Desktop/Dissertaion/China Data/Instruments/steel price index.xlsx",
    sheet_name='Sheet1')

# Convert the steel price data to a DataFrame
steel_price = pd.DataFrame(steel_price_file)

In [11]:
################ Construct differentiation IVs
# Key idea is to express the position of a brand in product characteristic space

# Define funciton to construct differentiation IVs

def construct_differentiation_ivs(market_share, threshold_std=1.0):
    """
    Construct IVs using positional indexing (6th col as mass, 7th as power)
    
    Parameters:
    -----------
    market_share : DataFrame
        Must have at least 7 columns with:
        - 6th column as mass
        - 7th column as power
    threshold_std : float
        Number of standard deviations for local difference threshold
    
    Returns:
    --------
    DataFrame with original data plus 6 new IV columns
    """
    
    # Create working copy
    df = market_share.copy()
    
    # Get columns by position
    mass_col = df.columns[6]
    power_col = df.columns[7]
    
    print(f"Using columns: {mass_col} as mass, {power_col} as power")
    
    # Calculate IVs
    
    ## Sum of rivals
    df['sum_rival_mass'] = df.groupby('market_id')[mass_col].transform('sum') - df[mass_col]
    df['sum_rival_power'] = df.groupby('market_id')[power_col].transform('sum') - df[power_col]
    
    ## Euclidean distance (vectorized)
    def euclidean(group, col):
        vals = group[col].values.astype(np.float32)
        diffs = np.subtract.outer(vals, vals)
        np.fill_diagonal(diffs, 0)
        return np.square(diffs).sum(axis=1)
    
    df['euclidean_mass'] = df.groupby('market_id').apply(lambda x: euclidean(x, mass_col)).explode().values
    df['euclidean_power'] = df.groupby('market_id').apply(lambda x: euclidean(x, power_col)).explode().values
    
    ## Local difference
    mass_thresh = threshold_std * df[mass_col].std()
    power_thresh = threshold_std * df[power_col].std()
    
    def local_diff(group, col, threshold):
        vals = group[col].values.astype(np.float32)
        diffs = np.abs(np.subtract.outer(vals, vals))
        np.fill_diagonal(diffs, 0)
        return (diffs < threshold).sum(axis=1)
    
    df['local_mass'] = df.groupby('market_id').apply(lambda x: local_diff(x, mass_col, mass_thresh)).explode().values
    df['local_power'] = df.groupby('market_id').apply(lambda x: local_diff(x, power_col, power_thresh)).explode().values
    
    return df

# Construct IVs
market_share_with_ivs = construct_differentiation_ivs(market_share, threshold_std=1)

# Display results
print(market_share_with_ivs)


Using columns: mass as mass, power as power


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16308\1197148182.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['euclidean_mass'] = df.groupby('market_id').apply(lambda x: euclidean(x, mass_col)).explode().values
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16308\1197148182.py:47: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['euclidean_power'] = df.groupby('market_id').apply(lambda x: euclidean(x, power_col

       year      type province brand     model fuel_type         mass  \
0      2019  国产新能源乘用车      上海市  东风风行     景逸S50       BEV  2036.000000   
1      2019  国产新能源乘用车      上海市    丰田       卡罗拉      PHEV  1975.000000   
2      2019  国产新能源乘用车      上海市    丰田        雷凌      PHEV  1975.000000   
3      2019  国产新能源乘用车      上海市    云度      云度π1       BEV  1785.000000   
4      2019  国产新能源乘用车      上海市    云度      云度π3       BEV  1845.000000   
...     ...       ...      ...   ...       ...       ...          ...   
79285  2023   国产燃油乘用车     黑龙江省    风神        皓极      PHEV  2093.000000   
79286  2023   国产燃油乘用车     黑龙江省   马自达  马自达CX-30       Gas  1880.750000   
79287  2023   国产燃油乘用车     黑龙江省   马自达   马自达CX-4       Gas  1977.363636   
79288  2023   国产燃油乘用车     黑龙江省   马自达   马自达CX-5       Gas  2018.479532   
79289  2023   国产燃油乘用车     黑龙江省   马自达   马自达CX-8       Gas  2441.315789   

            power  sales  weighted_Avg_Price  ...  product_id  market_id  \
0       90.000000      1                7.49  .

In [12]:
################ Construct exogenous cost-shifters

# Convert Year column into year in steel price data
steel_price['year'] = pd.to_datetime(steel_price['Year']).dt.year

# Drop 'Year' column as it's no longer needed
steel_price.drop(columns=['Year'], inplace=True)

# Rename columns for clarity
steel_price.rename(columns={'China: Steel Composite Price Indices:Annual:Average': 'steel_price_index'}, inplace=True)

# Use the mass of a model interacted with the steel price index as an exogenous cost-shifter
def construct_cost_shifters(market_share, steel_price):
    """
    Construct exogenous cost-shifters using weight and steel price index.
    
    Parameters:
    -----------
    market_share : DataFrame
        Must have a 'mass' column
    steel_price : DataFrame
        Must have a 'steel_price_index' column
    
    Returns:
    --------
    DataFrame with original data plus cost-shifter column
    """
    
    # Ensure steel price is aligned with market share data
    steel_price = steel_price.set_index('year')  # Assuming 'date' is the index
    
    # Merge on year
    merged = market_share.merge(steel_price, on='year', how='left')
    
    # Create cost-shifter as weight * steel price index
    merged['cost_shifter'] = merged['mass'] * merged['steel_price_index']
    
    return merged

# Construct cost-shifters
market_share_with_ivs_2 = construct_cost_shifters(market_share_with_ivs, steel_price)

In [13]:
################ Construct charging station IVs

# Use 1 period lag of charging station stock as an instrument
# Bartick Instrument: interacting national stock with local number of department stores

# Export market share data with IVs and cost-shifters
market_share_with_ivs_2.to_csv('data_with_IV.csv', index=False)